# Notebook 2: French data preparation and cross-lingual test
Run top to bottom on **Google Colab**, **after Notebook 1 has finished** (it saves `english_crosslingual_artifacts.joblib` to Drive).
The English models are NOT refit or retuned here: they are applied to French as they are.

In [ ]:
!pip -q install sentence-transformers huggingface_hub

import os, re, ast, joblib
import numpy as np, pandas as pd
from google.colab import drive
from sklearn.metrics import f1_score

drive.mount("/content/drive")
CKPT = "/content/drive/MyDrive/ML_Checkpoints"
os.makedirs(CKPT, exist_ok=True)
print("Files on Drive:", sorted(os.listdir(CKPT)))

## 1. Load the multilingual dataset and clean the French subset

In [ ]:
splits = {"train": "train_dataset.csv", "validation": "validation_dataset.csv", "test": "test_dataset.csv"}
raw = {k: pd.read_csv("hf://datasets/AnasAlokla/multilingual_go_emotions/" + v) for k, v in splits.items()}

LABEL_NAMES = [
    "admiration","amusement","anger","annoyance","approval","caring",
    "confusion","curiosity","desire","disappointment","disapproval",
    "disgust","embarrassment","excitement","fear","gratitude","grief",
    "joy","love","nervousness","optimism","pride","realization","relief",
    "remorse","sadness","surprise","neutral"
]
GROUP_MAP = {
    "admiration":"positive","amusement":"positive","approval":"positive",
    "caring":"positive","desire":"positive","excitement":"positive",
    "gratitude":"positive","joy":"positive","love":"positive",
    "optimism":"positive","pride":"positive","relief":"positive",
    "anger":"negative","annoyance":"negative","disappointment":"negative",
    "disapproval":"negative","disgust":"negative","embarrassment":"negative",
    "fear":"negative","grief":"negative","nervousness":"negative",
    "remorse":"negative","sadness":"negative",
    "confusion":"ambiguous","curiosity":"ambiguous",
    "realization":"ambiguous","surprise":"ambiguous",
    "neutral":"neutral"
}

def labels_to_groups_all(label_ids):
    names = [LABEL_NAMES[i] for i in label_ids]
    groups = [GROUP_MAP[n] for n in names]
    return list(dict.fromkeys(groups))

def process_split(df):
    """Keep rows whose labels collapse to ONE group; set the rest aside as conflicts."""
    df = df.copy()
    df["all_groups"] = df["labels"].apply(labels_to_groups_all)
    df["num_distinct_groups"] = df["all_groups"].apply(len)
    clean = df[df["num_distinct_groups"] == 1].copy()
    clean["group_label"] = clean["all_groups"].apply(lambda g: g[0])
    conflict = df[df["num_distinct_groups"] > 1].copy()
    return clean, conflict

def clean_text(text):
    text = re.sub(r"\[NOM\]|\[NAME\]|\[RELIGION\]", "", text)   # placeholder tokens (French uses [NOM])
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def prep_french(raw_df, split_name):
    df = raw_df[raw_df["language"] == "fr"].copy()
    print(f"{split_name}: {len(df)} French rows | null text: {df['text'].isnull().sum()} | "
          f"duplicate rows: {df.duplicated().sum()}")
    df = df[df["text"].notnull()].copy()
    df["clean_text"] = df["text"].apply(clean_text)
    df = df[df["clean_text"].str.strip() != ""].copy()
    df["labels"] = df["labels"].apply(ast.literal_eval)
    clean, conflict = process_split(df)
    print(f"{split_name}: kept {len(clean)} single-group rows, set aside {len(conflict)} conflicting rows")
    print(clean["group_label"].value_counts().to_string(), "\n")
    return clean, conflict

train_clean_f, train_conflict_f = prep_french(raw["train"], "train")
val_clean_f,   val_conflict_f   = prep_french(raw["validation"], "validation")
test_clean_f,  test_conflict_f  = prep_french(raw["test"], "test")

## 2. Save the cleaned French TEST set to Drive

In [ ]:
test_clean_f[["id", "text", "clean_text", "group_label"]].to_csv(f"{CKPT}/french_test_clean.csv", index=False)
print("Saved:", os.path.exists(f"{CKPT}/french_test_clean.csv"), "| rows:", len(test_clean_f))

## 3. Cross-lingual test: English-trained, English-tuned models applied to French (no refitting)

In [ ]:
from sentence_transformers import SentenceTransformer

# English side, saved by Notebook 1
art = joblib.load(f"{CKPT}/english_crosslingual_artifacts.joblib")
tfidf, eng_models, en_preds = art["tfidf"], art["models"], art["test_preds"]
en = pd.DataFrame({"id": art["test_ids"], "group_label": art["test_labels"]})
en["en_pos"] = np.arange(len(en))

# French side
fr = pd.read_csv(f"{CKPT}/french_test_clean.csv").dropna(subset=["clean_text"])
fr["id"] = fr["id"].astype(str)
fr["orig_id"] = fr["id"].str[2:]            # strip the 2-letter language prefix (e.g. "FR")
print("English ids:", en["id"].head(3).tolist(), "| French ids (stripped):", fr["orig_id"].head(3).tolist())

# Pair the same comment in both languages by its original id
m = en.merge(fr[["orig_id", "group_label", "clean_text"]], left_on="id", right_on="orig_id",
             suffixes=("_en", "_fr"))
print(len(m), "paired |", len(en), "English rows |", len(fr), "French rows | ids unique:", m["id"].is_unique)
if len(m) < 0.9 * len(fr):
    print("WARNING: fewer than 90% of French rows matched. The id format is probably different: compare the two id lists above.")
n_disagree = int((m["group_label_en"] != m["group_label_fr"]).sum())
print("label disagreements between English and French:", n_disagree)
m = m[m["group_label_en"] == m["group_label_fr"]].reset_index(drop=True)
print(int(m["clean_text"].str.contains("â€|Ã").sum()), "French rows with possible encoding errors")

# Embed French with the SAME frozen LaBSE (a GPU runtime makes this much faster)
labse = SentenceTransformer("sentence-transformers/LaBSE")
E_fr = labse.encode(list(m["clean_text"]), batch_size=64, show_progress_bar=True)
np.save(f"{CKPT}/labse_french_test.npy", E_fr)

y, pos = m["group_label_en"].to_numpy(), m["en_pos"].to_numpy()
X_fr_tfidf = tfidf.transform(m["clean_text"])
fr_inputs = {
    "LaBSE + Logistic Regression":  E_fr,
    "TF-IDF + Logistic Regression": X_fr_tfidf,
    "TF-IDF + Naive Bayes":         X_fr_tfidf,
}

# Paired bootstrap on the same comments in both languages
rng = np.random.default_rng(42)
samples = [rng.integers(0, len(y), len(y)) for _ in range(1000)]
classes = ["positive", "negative", "ambiguous", "neutral"]
french_preds, rows = {}, []
for name, X_fr in fr_inputs.items():
    p_fr = eng_models[name].predict(X_fr)
    p_en = np.asarray(en_preds[name])[pos]
    french_preds[name] = p_fr
    f_en, f_fr = f1_score(y, p_en, average="macro"), f1_score(y, p_fr, average="macro")
    d = [f1_score(y[i], p_en[i], average="macro") - f1_score(y[i], p_fr[i], average="macro") for i in samples]
    lo, hi = np.percentile(d, [2.5, 97.5])
    rows.append({"Model": name, "English macro-F1": round(f_en, 4), "French macro-F1": round(f_fr, 4),
                 "Drop (EN-FR)": round(f_en - f_fr, 4), "Drop 95% CI": f"[{lo:.3f}, {hi:.3f}]"})
    print(name, "| per-class F1 EN:", f1_score(y, p_en, labels=classes, average=None).round(3),
          "FR:", f1_score(y, p_fr, labels=classes, average=None).round(3), classes)

fr_table = pd.DataFrame(rows)
fr_table.to_csv(f"{CKPT}/cross_lingual_results.csv", index=False)
display(fr_table)

# Row-level predictions, for the error analysis
out = m[["id", "group_label_en", "clean_text"]].rename(columns={"group_label_en": "true_label", "clean_text": "french_text"})
for name in fr_inputs:
    out[f"pred_EN | {name}"] = np.asarray(en_preds[name])[pos]
    out[f"pred_FR | {name}"] = french_preds[name]
out.to_csv(f"{CKPT}/french_predictions.csv", index=False)
print("Saved cross_lingual_results.csv and french_predictions.csv")